In [1]:
import dlt, duckdb, csv, requests
from io import StringIO

In [2]:
HIVDB_DATASETS = {
    "pi": "https://hivdb.stanford.edu/download/GenoPhenoDatasets/PI_DataSet.txt",
    "nrti": "https://hivdb.stanford.edu/download/GenoPhenoDatasets/NRTI_DataSet.txt",
    "nnrti": "https://hivdb.stanford.edu/download/GenoPhenoDatasets/NNRTI_DataSet.txt",
    "ini": "https://hivdb.stanford.edu/download/GenoPhenoDatasets/INI_DataSet.txt",
    "cai": "https://hivdb.stanford.edu/download/GenoPhenoDatasets/CAI_DataSet.txt"
}

In [3]:
@dlt.source
def hivdb_source(username, password):
    session = requests.Session()
    session.auth = (username, password)

    for name, url in HIVDB_DATASETS.items():

        @dlt.resource(name=name, write_disposition="replace")
        def fetch_dataset(_url=url, _name=name):
            response = session.get(_url)
            response.raise_for_status()
            reader = csv.DictReader(StringIO(response.text), delimiter="\t")
            for row in reader:
                yield row

        yield fetch_dataset


pipeline = dlt.pipeline(
    pipeline_name="hivdb_resistance",
    destination="duckdb",
    dataset_name="hivdb",
)

load_info = pipeline.run(hivdb_source(
    username="your_username",
    password="your_password",
))
print(load_info)

Pipeline hivdb_resistance load step completed in 3.32 seconds
1 load package(s) were loaded to destination duckdb and into dataset hivdb
The duckdb destination used duckdb:////Users/benzenesea/Desktop/hiv-resistance-analytics/ingestion/hivdb_resistance.duckdb location to store data
Load package 1774664506.641375 is LOADED and contains no failed jobs


In [4]:
print(pipeline.dataset().pi.df())

      seq_id  fpv  atv   idv   lpv   nfv  sqv  tpv  drv p1  ... p93 p94 p95  \
0      12862  0.8   NA   1.2    NA  24.7  0.9   NA   NA  -  ...   -   -   -   
1      13259  0.1   NA   2.9    NA  12.2  1.4   NA   NA  -  ...   -   -   -   
2      12863  3.0   NA   2.8    NA   2.2  1.0   NA   NA  -  ...   -   -   -   
3      13257  0.1   NA   2.3    NA   8.1  1.1   NA   NA  -  ...   -   -   -   
4      13258  0.1   NA   3.0    NA  12.8  1.2   NA   NA  -  ...   -   -   -   
...      ...  ...  ...   ...   ...   ...  ...  ...  ... ..  ...  ..  ..  ..   
2166   15750  4.3  1.2  18.0  23.0   3.8  0.4   NA   NA  -  ...   L   -   -   
2167   60639  0.3   NA   0.4   0.3   0.4  0.3   NA   NA  -  ...   -   -   -   
2168   44100  1.3   NA   1.9    NA   3.2  2.6   NA   NA  -  ...   -   -   -   
2169  145194  1.0  1.0   1.8   0.8   2.4  1.3  1.7  1.3  -  ...   L   -   -   
2170  109390  0.7  0.9   0.7   0.7   1.0  1.1  1.4  0.9  -  ...   -   -   -   

     p96 p97 p98 p99                               

In [5]:
print(pipeline.dataset().nrti.df())

      seq_id _3_tc  abc   azt d4_t  ddi  tdf p1 p2 p3  ... p234 p235 p236  \
0      54191   9.0  2.4   0.6  1.8  1.6  1.9  -  -  -  ...    -    -    -   
1      60631   9.7  3.0   0.5  1.8  2.1  2.1  -  -  -  ...    -    -    -   
2      60636   100  8.5   0.3  0.9  2.4  0.4  -  -  -  ...    -    -    -   
3      54192   6.3  1.6   0.5  1.4  1.3  1.8  -  -  -  ...    -    -    -   
4      60630   1.0  0.9   0.8  0.8  0.8  0.9  -  -  -  ...    -    -    -   
...      ...   ...  ...   ...  ...  ...  ... .. .. ..  ...  ...  ...  ...   
1862  205635   100  5.4   1.4  1.3  2.0  0.6  -  -  -  ...    -    -    -   
1863   86891   100  6.1   7.7  3.6  2.5  1.4  -  -  -  ...    -    -    -   
1864  112976  96.1   NA  50.0  3.1  2.3  1.5  .  .  .  ...    -    -    -   
1865  614965   0.9  0.8   0.6  1.1  1.0  0.8  -  -  -  ...    -    -    -   
1866  148042   1.0  1.0   0.6  0.7  1.0  0.7  -  -  -  ...    -    -    -   

     p237 p238 p239 p240                                      comp_mut_list

In [6]:
print(pipeline.dataset().nnrti.df())

      seq_id   efv   nvp  etr  rpv dor p1 p2 p3 p4  ... p312 p313 p314 p315  \
0     205675  25.0   100  1.2  1.4  NA  -  -  -  -  ...    .    .    .    .   
1     205679   0.4   0.4  0.3  0.3  NA  -  -  -  -  ...    .    .    .    .   
2     216543    NA    NA  1.2   NA  NA  -  -  -  -  ...    -    -    -    -   
3     216546    NA    NA  0.9   NA  NA  -  -  -  -  ...    -    -    -    -   
4     216550    NA    NA  9.5   NA  NA  -  -  -  -  ...    -    -    -    -   
...      ...   ...   ...  ...  ...  .. .. .. .. ..  ...  ...  ...  ...  ...   
2265   44000  15.0   100   NA   NA  NA  -  -  -  -  ...    .    .    .    .   
2266   44001  15.0   100   NA   NA  NA  -  -  -  -  ...    -    -    -    -   
2267  149233   1.3   5.2   NA   NA  NA  -  -  -  -  ...    .    .    .    .   
2268   74921  81.0  94.0   NA   NA  NA  -  -  -  -  ...    .    .    .    .   
2269   38859   0.2   0.2   NA   NA  NA  -  -  -  -  ...    -    -    -    -   

     p316 p317 p318                                

In [7]:
print(pipeline.dataset().ini.df())

     seq_id   ral   evg  dtg  bic  cab p1 p2 p3 p4  ... p282 p283 p284 p285  \
0    219526   0.9   7.6  0.3   NA   NA  -  -  -  -  ...    -    -    -    -   
1    219527   1.1   4.1  1.0   NA   NA  -  -  -  -  ...    -    -    -    -   
2    219528  15.0   4.9  0.4   NA   NA  -  -  -  -  ...    -    -    -    -   
3    219529  37.0  94.0  1.5   NA   NA  -  -  -  -  ...    -    -    -    -   
4    219530  29.0  92.0  1.1   NA   NA  -  -  -  -  ...    -    -    -    -   
..      ...   ...   ...  ...  ...  ... .. .. .. ..  ...  ...  ...  ...  ...   
841  308284   0.8   1.1   NA   NA   NA  -  -  -  -  ...    -    -    -    -   
842  773603    NA  19.0   NA  1.1  1.8  -  -  -  -  ...    -    -    -    -   
843  308567   0.7   1.3   NA   NA   NA  -  -  -  -  ...    -    -    -    -   
844  308446   1.1   1.3   NA   NA   NA  -  -  -  -  ...    -    -    -    -   
845  467173    NA    NA  0.7  1.5   NA  -  -  -  -  ...    -    G    -    -   

    p286 p287 p288                                 

In [8]:
print(pipeline.dataset().cai.df())

    seq_id   len p1 p2 p3 p4 p5 p6 p7 p8  ... p225 p226 p227 p228 p229 p230  \
0   749359   6.0  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
1   749360  22.0  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
2   749361  24.0  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
3   749362  32.0  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
4   749363  62.0  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
..     ...   ... .. .. .. .. .. .. .. ..  ...  ...  ...  ...  ...  ...  ...   
66  749356   7.1  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
67  759420   0.6  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
68  759419   100  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
69  759401   100  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   
70  759404   100  -  -  -  -  -  -  -  -  ...    -    -    -    -    -    -   

   p231      comp_mut_list       _dlt_load_id      

In [9]:
print(pipeline.working_dir)
print(pipeline.destination)

/Users/benzenesea/.dlt/pipelines/hivdb_resistance
<dlt.destinations.duckdb(destination_type='duckdb', staging_dataset_name_layout='%s_staging', enable_dataset_name_normalization=True, info_tables_query_threshold=1000, truncate_tables_on_staging_destination_before_load=True, local_dir='/Users/benzenesea/Desktop/hiv-resistance-analytics/ingestion', pipeline_name='hivdb_resistance', pipeline_working_dir='/Users/benzenesea/.dlt/pipelines/hivdb_resistance', create_indexes=False)>


In [10]:
import os
duckdb_path = os.path.join(pipeline.working_dir, 'duckdb', 'hivdb_resistance.duckdb')
print(f"Path: {duckdb_path}")
print(f"Exists: {os.path.exists(duckdb_path)}")

Path: /Users/benzenesea/.dlt/pipelines/hivdb_resistance/duckdb/hivdb_resistance.duckdb
Exists: False


In [11]:
import duckdb
conn = duckdb.connect('/Users/benzenesea/Desktop/hiv-resistance-analytics/ingestion/hivdb_resistance.duckdb')
print('Tables:', conn.execute('SHOW TABLES').fetchall())
print('Schemas:', conn.execute('SELECT schema_name FROM information_schema.schemata').fetchall())

Tables: [('test_read',)]
Schemas: [('hivdb',), ('main',), ('information_schema',), ('main',), ('pg_catalog',), ('main',)]


In [12]:
print(conn.execute("SELECT table_schema, table_name FROM information_schema.tables").fetchall())

[('hivdb', 'cai'), ('hivdb', 'ini'), ('hivdb', 'insti'), ('hivdb', 'nnrti'), ('hivdb', 'nrti'), ('hivdb', 'pi'), ('hivdb', '_dlt_loads'), ('hivdb', '_dlt_pipeline_state'), ('hivdb', '_dlt_version'), ('main', 'test_read')]


In [13]:
print(conn.execute("SELECT * FROM hivdb.pi LIMIT 5").fetchdf())

  seq_id  fpv atv  idv lpv   nfv  sqv tpv drv p1  ... p93 p94 p95 p96 p97 p98  \
0  12862  0.8  NA  1.2  NA  24.7  0.9  NA  NA  -  ...   -   -   -   -   -   -   
1  13259  0.1  NA  2.9  NA  12.2  1.4  NA  NA  -  ...   -   -   -   -   -   -   
2  12863  3.0  NA  2.8  NA   2.2  1.0  NA  NA  -  ...   -   -   -   -   -   -   
3  13257  0.1  NA  2.3  NA   8.1  1.1  NA  NA  -  ...   -   -   -   -   -   -   
4  13258  0.1  NA  3.0  NA  12.8  1.2  NA  NA  -  ...   -   -   -   -   -   -   

  p99                 comp_mut_list       _dlt_load_id         _dlt_id  
0   -  D30N, M46I, R57G, L63P, N88D  1774664506.641375  tSFQsIPOriSKVg  
1   -  M46L, R57G, L63P, V77I, N88S  1774664506.641375  IliCNd2lp/Ziuw  
2   -  M46I, R57G, L63P, V82T, I84V  1774664506.641375  AuKC6RnZz3RFPg  
3   -        M46L, R57G, L63P, N88S  1774664506.641375  jzXNgrmIZVvt3Q  
4   -        R57G, L63P, V77I, N88S  1774664506.641375  eEzRTo0TWBEUxA  

[5 rows x 111 columns]


In [14]:
conn.close()